# Lesson 07 Lab — Masks and Reduction Identities

**Puzzle:** When masked loads, other values, and reduction algebra change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates masked loads, other values, and reduction algebra and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

A mask does more than prevent an invalid address: the value supplied for an inactive lane enters subsequent tensor algebra. Sum needs zero, max needs negative infinity, and min needs positive infinity. The correct identity depends on the downstream operation.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["masked loads, other values, and reduction algebra"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

Padding an all-negative max reduction with zero silently returns a plausible but impossible answer.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 7
LESSON_TITLE = 'Masks and Reduction Identities'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260820
}


## 5. Freeze the experiment

**Experiment:** Compare correct -inf padding with incorrect zero padding on 1,024 all-negative rows.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.0,
  "secondary": 1.0080831050872803,
  "max_abs_error": 0.0,
  "passed": true,
  "details": {
    "corrupt_rows": 1024,
    "tail": 24
  }
}
Using -inf as the masked max identity produced 0.00e+00 error; padding with zero corrupted 1024 all-negative rows.


## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Correct-mask error | 0.000e+00 |
| Zero-pad error | 1.008e+00 |
| Maximum absolute error | 0.000e+00 |
| Acceptance gate | true |


## 8. Explain without overclaiming

Using -inf as the masked max identity produced 0.00e+00 error; padding with zero corrupted 1024 all-negative rows.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Treat mask values as part of the mathematical specification, not as a memory-safety afterthought.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 7,
  "title": "Masks and Reduction Identities",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260820
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.0,
    "secondary": 1.0080831050872803,
    "max_abs_error": 0.0,
    "passed": true,
    "details": {
      "corrupt_rows": 1024,
      "tail": 24
    }
  },
  "analysis_en": "Using -inf as the masked max identity produced 0.00e+00 error; padding with zero corrupted 1024 all-negative rows.",
  "analysis_zh": "max reduction 的 mask 使用 -inf 时误差为 0.00e+00；错误地填 0 会破坏 1024 行全负输入。",
  "conclusion": "Treat mask values as part of the mathematical specification, not as a memory-safety afterthought."
}


## 10. Make the bounded decision

> Treat mask values as part of the mathematical specification, not as a memory-safety afterthought.

**Failure analysis:** Padding an all-negative max reduction with zero silently returns a plausible but impossible answer.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
